Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency

df_cc_calls = pd.read_csv("../../data/interim/cc_calls_cleaned.csv")

Settings for Pandas and visualization

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_style('whitegrid')
sns.set_palette('viridis')

Loading CC_Calls cleaned data

In [3]:
df_cc_calls = pd.read_csv('../../data/interim/cc_calls_cleaned.csv')
print(f"Shape: {df_cc_calls.shape}")
df_cc_calls.head()

Shape: (31636, 30)


,contact_id,call_date,direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_call_initiated_by,cc_questionnaire_completion,cc_chasing_response,cc_issues_within_questionnaire,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_contractor_sentiment,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,co_ref
0,6.255130e+11,2025-05-08,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,Customer,No,No,No,No,Yes,Yes,Yes,No,Yes,Dissatisfied,20.0,30.0,30.0,Yes,Yes,No,Yes,Yes,HV3323
1,5.910870e+11,2024-11-25,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,Agent,Yes,No,No,No,Yes,Yes,Yes,No,No,Dissatisfied,0.0,0.0,0.0,Yes,Yes,No,Yes,Yes,PJ7066
2,5.650910e+11,2024-10-23,IN_BOUND,Standard,Yes,No,No,No,Yes,No,Customer,No,No,No,No,Yes,No,Yes,No,No,Dissatisfied,20.0,60.0,40.0,Yes,Yes,No,Yes,Yes,DP6030
3,5.939750e+11,2025-01-13,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,Customer,Yes,No,No,No,Yes,Yes,Yes,No,Yes,Dissatisfied,20.0,60.0,40.0,Yes,Yes,Yes,Yes,Yes,AM2413
4,6.222820e+11,2025-03-19,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,Customer,No,No,Unknown,No,No,No,Yes,No,Yes,Dissatisfied,20.0,40.0,40.0,Yes,Yes,No,Yes,Yes,ED6707


Convert Yes/No/Unknown categorical columns to binary

In [4]:

# Yes = 1, No = 0, Unknown = 0 (no confirmed signal)
yes_no_cols = [
    'cc_care_package_discussed',
    'cc_urgency_getting_on_site',
    'cc_external_consultant',
    'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion',
    'cc_chasing_response',
    'cc_issues_within_questionnaire',
    'cc_login_issues',
    'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_pricing_mentioned',
    'cc_pricing_sentiment_impact',
    'cc_refund_discussed',
    'cc_contractor_suggest_leave',
    'cc_contractor_complained',
]
 
for col in yes_no_cols:
    df_cc_calls[col] = df_cc_calls[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})

Encoding Sentiment Column

In [5]:
# Dissatisfied = -1, Neutral = 0, Satisfied = 1
sentiment_map = {
    'Satisfied'    :  1,
    'Neutral'      :  0,
    'Not Discussed':  0,
    'Dissatisfied' : -1,
    'Unknown'      :  0,
}
 
df_cc_calls['cc_sentiment_encoded'] = df_cc_calls['cc_contractor_sentiment'].map(sentiment_map)
df_cc_calls['cc_dissatisfied_flag'] = (df_cc_calls['cc_contractor_sentiment'] == 'Dissatisfied').astype(int)

Encoding Direction Column

In [6]:
# IN_BOUND = 1, OUT_BOUND = 0
df_cc_calls['is_inbound'] = (df_cc_calls['direction'] == 'IN_BOUND').astype(int)

Feature engineering


In [7]:
# Sentiment improved or worsened during the call
df_cc_calls['sentiment_improved'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] > 
    df_cc_calls['cc_contractor_sentiment_start_score']
).astype(int)
 
df_cc_calls['sentiment_worsened'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] < 
    df_cc_calls['cc_contractor_sentiment_start_score']
).astype(int)
 
# Sentiment change score
df_cc_calls['sentiment_change'] = (
    df_cc_calls['cc_contractor_sentiment_end_score'] - 
    df_cc_calls['cc_contractor_sentiment_start_score']
)
 
# High risk call — multiple issues in one call
df_cc_calls['high_risk_call'] = (
    (df_cc_calls['cc_contractor_complained']          == 1) |
    (df_cc_calls['cc_contractor_suggest_leave']        == 1) |
    (df_cc_calls['cc_business_struggles_financial_hardship'] == 1) |
    (df_cc_calls['cc_refund_discussed']               == 1)
).astype(int)
 
# Dissatisfaction issue count per call
dissatisfaction_cols = [
    'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support',
    'cc_platform_issues',
    'cc_login_issues',
]
df_cc_calls['dissatisfaction_issue_count'] = df_cc_calls[dissatisfaction_cols].sum(axis=1)
 
# Pricing pressure flag — pricing mentioned and had sentiment impact
df_cc_calls['pricing_pressure_flag'] = (
    (df_cc_calls['cc_pricing_mentioned']        == 1) &
    (df_cc_calls['cc_pricing_sentiment_impact'] == 1)
).astype(int)

Aggregate Per Customer using co_ref

In [ ]:
agg_dict = {
    # Call counts
    'contact_id'                              : 'count',
    'is_inbound'                              : 'sum',
 
    # Sentiment
    'cc_sentiment_encoded'                    : 'mean',
    'cc_dissatisfied_flag'                    : 'max',
    'cc_contractor_sentiment_start_score'     : 'mean',
    'cc_contractor_sentiment_end_score'       : 'mean',
    'cc_contractor_sentiment_overall_score'   : 'mean',
 
    # Issue flags — max means 1 if it happened at least once
    'cc_customer_issues_concerns'             : 'max',
    'cc_business_struggles_financial_hardship': 'max',
    'cc_login_issues'                         : 'max',
    'cc_platform_issues'                      : 'max',
    'cc_dissatisfaction_time_to_complete'     : 'max',
    'cc_process_complexity_concerns'          : 'max',
    'cc_questions_harder_than_expected'       : 'max',
    'cc_dissatisfaction_support'              : 'max',
    'cc_pricing_mentioned'                    : 'max',
    'cc_pricing_sentiment_impact'             : 'max',
    'cc_refund_discussed'                     : 'max',
    'cc_contractor_suggest_leave'             : 'max',
    'cc_contractor_complained'                : 'max',
    'cc_urgency_getting_on_site'              : 'max',
    'cc_chasing_response'                     : 'max',
    'cc_questionnaire_completion'             : 'max',
    'sentiment_improved'         : 'max',
    'sentiment_worsened'         : 'max',
    'sentiment_change'           : 'mean',
    'high_risk_call'             : 'max',
    'dissatisfaction_issue_count': 'sum',
    'pricing_pressure_flag'      : 'max'
}
 
df_cc_calls_agg = df_cc_calls.groupby('co_ref').agg(agg_dict).reset_index()
 
# Rename columns clearly
df_cc_calls_agg.rename(columns={
    'contact_id'  : 'cc_total_calls',
    'is_inbound'  : 'cc_inbound_calls',
}, inplace=True)
 
print(f"Shape after aggregation: {df_cc_calls_agg.shape}")
df_cc_calls_agg.head()


Shape after aggregation: (14988, 30)


,co_ref,cc_total_calls,cc_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,sentiment_improved,sentiment_worsened,sentiment_change,high_risk_call,dissatisfaction_issue_count,pricing_pressure_flag
0,AA0584,1,0,0.0,0,50.0,50.0,80.0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0.0,0,0,0
1,AA0750,1,0,1.0,0,50.0,90.0,90.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,40.0,0,0,0
2,AA0784,2,0,0.5,0,50.0,65.0,80.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,15.0,0,0,0
3,AA0882,1,1,0.0,0,50.0,80.0,70.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,30.0,0,1,0
4,AA0923,1,1,1.0,0,50.0,90.0,90.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,40.0,0,0,0


Add Outbound Calls Column

In [9]:
df_cc_calls_agg['cc_outbound_calls'] = (
    df_cc_calls_agg['cc_total_calls'] - df_cc_calls_agg['cc_inbound_calls']
)

Final check for nulls and shape

In [10]:
print("Nulls:")
print(df_cc_calls_agg.isnull().sum()[df_cc_calls_agg.isnull().sum() > 0])
 
print(f"\nShape: {df_cc_calls_agg.shape}")
print(f"Unique customers: {df_cc_calls_agg['co_ref'].nunique()}")
df_cc_calls_agg.describe()

Nulls:
Series([], dtype: int64)

Shape: (14988, 31)
Unique customers: 14988


,cc_total_calls,cc_inbound_calls,cc_sentiment_encoded,cc_dissatisfied_flag,cc_contractor_sentiment_start_score,cc_contractor_sentiment_end_score,cc_contractor_sentiment_overall_score,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,cc_login_issues,cc_platform_issues,cc_dissatisfaction_time_to_complete,cc_process_complexity_concerns,cc_questions_harder_than_expected,cc_dissatisfaction_support,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,cc_urgency_getting_on_site,cc_chasing_response,cc_questionnaire_completion,sentiment_improved,sentiment_worsened,sentiment_change,high_risk_call,dissatisfaction_issue_count,pricing_pressure_flag,cc_outbound_calls
count,14988.000000,14988.000000,14988.00000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000,14988.000000
mean,2.110755,0.489992,0.48621,0.054911,52.604799,79.397293,81.632834,0.176608,0.057246,0.090873,0.126568,0.082399,0.167200,0.007740,0.040566,0.213704,0.062317,0.011209,0.046170,0.123432,0.167401,0.364291,0.315252,0.925140,0.015212,26.792494,0.173939,0.602349,0.062050,1.620763
std,1.602405,1.068655,0.46980,0.227813,10.056363,13.114121,9.400232,0.381349,0.232319,0.287438,0.332499,0.274981,0.373167,0.087636,0.197289,0.409934,0.241738,0.105281,0.209861,0.328943,0.373346,0.481247,0.464632,0.263174,0.122400,13.471796,0.379070,1.242150,0.241254,1.405135
min,1.000000,0.000000,-1.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-50.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.00000,0.000000,50.000000,75.000000,80.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,20.000000,0.000000,0.000000,0.000000,1.000000
50%,2.000000,0.000000,0.50000,0.000000,50.000000,80.000000,82.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,30.000000,0.000000,0.000000,0.000000,1.000000
75%,3.000000,1.000000,1.00000,0.000000,50.000000,90.000000,87.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,37.500000,0.000000,1.000000,0.000000,2.000000
max,60.000000,60.000000,1.00000,1.000000,100.000000,100.000000,100.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,70.000000,1.000000,27.000000,1.000000,16.000000


Saving final features dataset

In [11]:
df_cc_calls_agg.to_csv('../../data/interim/cc_calls_features.csv', index=False)
print("Saved => data/interim/cc_calls_features.csv")

Saved => data/interim/cc_calls_features.csv


### Hypothesis

Hypothesis 1 => Customers with higher number of issues are more likely to churn

Hypothesis 2 => Customers with lower sentiment scores tend to churn

Hypothesis 3 => Customers who complain more frequently are more likely to churn

Hypothesis 4 => Customers with fewer interactions (calls) are more likely to churn

Hypothesis 5 => Negative experience over time (low sentiment + high issues) increases churn probability